In [ ]:
# --------------------------------------------------------------------------------
# GenoBench — Paired Associate Learning
# Track: Learning | Sub-Ability: Associative Learning
#
# Can the model learn and recall arbitrary paired associations?
# Pre/post learning framework: memorize novel word-meaning pairs, then recall.
# Difficulty grid: pair complexity × num pairs × 3 seeds = 27 items
#
# 📚 LEARNING RESOURCES
# Quick Start: https://github.com/Kaggle/kaggle-benchmarks/blob/ci/quick_start.md
# Cookbook:    https://github.com/Kaggle/kaggle-benchmarks/blob/ci/cookbook.md
# --------------------------------------------------------------------------------

import kaggle_benchmarks as kbench
import pandas as pd
import re
import time
import glob

In [ ]:
# --------------------------------------------------------------------------------
# STEP 1: LOAD DATASET
# 27 items: 3 complexity levels (simple, compound, relational)
#         × 3 evidence levels (few=4, mid=8, many=12 pairs)
#         × 3 random seeds
# Dataset: https://www.kaggle.com/datasets/eugenio0/paired-associate-benchmark
# --------------------------------------------------------------------------------

candidates = glob.glob('/kaggle/input/**/paired_associate_dataset.csv', recursive=True)
csv_path = candidates[0] if candidates else '/kaggle/input/paired_associate_dataset.csv'
dataset = pd.read_csv(csv_path)
print(f'Dataset: {len(dataset)} items from {csv_path}')

In [ ]:
# --------------------------------------------------------------------------------
# STEP 2: HELPERS
# Strip reasoning-model thinking blocks and extract short answers.
# --------------------------------------------------------------------------------

def strip_thinking(text: str) -> str:
    """Remove <think>...</think> blocks from reasoning model output."""
    if '</think>' in text:
        return text.split('</think>', 1)[1].strip()
    return text.strip()

def extract_short_answer(response: str) -> str:
    """Extract a short answer (last short line, stripped of formatting)."""
    text = strip_thinking(response)
    text = re.sub(r'[`*]', '', text)
    lines = [l.strip() for l in text.strip().split('\n') if l.strip()]
    if not lines:
        return ''
    for line in reversed(lines):
        clean = line.strip('"\'\u2018\u2019\u201c\u201d').rstrip('.!?')
        if len(clean) <= 60:
            return clean.lower().strip()
    return lines[-1].strip('"\'\u2018\u2019\u201c\u201d').rstrip('.!?').lower().strip()


# --- Prompt templates ---

PRE_PROMPT = """Here is a vocabulary from an unknown language:

{material}

What does "{test_input}" mean?

Reply with ONLY the meaning (e.g., "red" or "very blue"). No explanation."""

STUDY_PROMPT = """Here is a vocabulary from an unknown language:

{material}

Create a systematic study guide for memorizing this vocabulary:
1. Group related words (e.g., words with similar sounds or related meanings).
2. Identify any patterns in the word-meaning mappings.
3. Create a mnemonic or memory aid for each word.
4. If there are modifier words (marked with -), explain how they combine with base words.
5. List every word-meaning pair in a clear summary table.

Show all work."""

POST_PROMPT = """You previously studied a vocabulary from an unknown language and made these notes:

--- YOUR NOTES ---
{notes}
--- END NOTES ---

Original vocabulary:
{material}

What does "{test_input}" mean?

Reply with ONLY the meaning (e.g., "red" or "very blue"). No explanation."""

In [ ]:
# --------------------------------------------------------------------------------
# STEP 3: DEFINE THE TASK
# The @task decorator turns a standard Python function into a Benchmark task.
# For each of 27 items, the model:
#   A. PRE-TEST  — answer cold (baseline, no study)
#   B. STUDY     — create a memorization guide from the vocabulary
#   C. POST-TEST — answer again with its own study notes
# Returns post-learning accuracy as a float (0.0–1.0).
# --------------------------------------------------------------------------------

@kbench.task(
    name='GenoBench-paired_associate',
    description='Pre/post learning: can the model memorize novel word-meaning pairs?'
)
def paired_associate(llm) -> float:
    model_name = str(llm)
    correct = 0
    total = len(dataset)

    for _, row in dataset.iterrows():
        material = row['material']
        test_input = row['test_input']
        expected = row['expected'].lower().strip()
        difficulty = row['difficulty_label']

        # A. PRE-TEST: baseline without study
        pre_raw = llm.prompt(PRE_PROMPT.format(material=material, test_input=test_input))
        pre_answer = extract_short_answer(pre_raw)
        pre_correct = pre_answer == expected

        # B. STUDY: model creates its own memorization guide
        study_raw = llm.prompt(STUDY_PROMPT.format(material=material))
        notes = strip_thinking(study_raw)

        # C. POST-TEST: recall with study notes
        post_raw = llm.prompt(POST_PROMPT.format(
            notes=notes, material=material, test_input=test_input
        ))
        post_answer = extract_short_answer(post_raw)
        post_correct = post_answer == expected

        if post_correct:
            correct += 1

        # Log progress
        pre_tag = 'Y' if pre_correct else 'N'
        post_tag = 'Y' if post_correct else 'N'
        print(f'[{difficulty:18s}] "{test_input}" -> "{expected}"  '
              f'pre="{pre_answer}"({pre_tag})  post="{post_answer}"({post_tag})')

        # Assert post-learning correctness
        kbench.assertions.assert_equal(post_answer, expected)

    score = correct / total
    print(f'\nPost-learning accuracy: {correct}/{total} = {score:.1%}')
    return score

In [ ]:
# --------------------------------------------------------------------------------
# STEP 4: RUN THE TASK
# We use `kbench.llm` as a placeholder. This allows Kaggle to automatically swap
# in different models later when you use the "Add Models" button in the UI.
# --------------------------------------------------------------------------------

paired_associate.run(kbench.llm)

# Note: To test a specific model locally, you can use the dictionary lookup:
# paired_associate.run(kbench.llms["google/gemini-2.0-flash"])

In [ ]:
# --------------------------------------------------------------------------------
# STEP 5: NEXT STEPS
# 1. Click "Save Task" (top right) to publish to the leaderboard.
# 2. Add more models via the "Add Models" button to compare learning ability.
# 3. See the full analysis notebook (paired_associate.ipynb) for detailed
#    results breakdowns, study notes inspection, and visualizations.
# --------------------------------------------------------------------------------